# Experiment 03: Metric

Card: [exp03_metric.md](../../experiments/exp03_metric.md)

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.float_format", "{:,.3f}".format)
# Nothing here scales with the data; a smoke run only skips saving the figure
FIGURE = None if os.environ.get("SMOKE") == "1" else "../../figures/exp03_metric.png"
sns.set_theme(style="whitegrid")

## RMSLE on single predictions

In [ ]:
def rmsle(y, p):
    return np.sqrt(np.mean((np.log1p(p) - np.log1p(y)) ** 2))


y = np.array([100.0, 100.0, 10_000.0, 10_000.0, 0.0])
cases = {
    "+10% on small value": (y[0], 110.0),
    "-10% on small value": (y[1], 90.0),
    "+10% on large value": (y[2], 11_000.0),
    "absolute +1,000 on large": (y[3], 11_000.0),
    "absolute +1,000 on small": (y[0], 1_100.0),
    "predict 5 when truth is 0": (y[4], 5.0),
}
display(
    pd.DataFrame(
        {
            k: {"y": a, "pred": b, "per-row RMSLE": rmsle(np.array([a]), np.array([b]))}
            for k, (a, b) in cases.items()
        }
    ).T
)

,y,pred,per-row RMSLE
+10% on small value,100.000,110.000,0.094
-10% on small value,100.000,90.000,0.104
+10% on large value,"10,000.000","11,000.000",0.095
"absolute +1,000 on large","10,000.000","11,000.000",0.095
"absolute +1,000 on small",100.000,"1,100.000",2.389
predict 5 when truth is 0,0.000,5.000,1.792


In [ ]:
# Cost of one prediction across the whole range, for three true values
pred = np.logspace(-1, 6, 500)

fig, ax = plt.subplots(figsize=(9, 4))
for truth, color in [(0, "tab:red"), (100, "tab:blue"), (10_000, "tab:green")]:
    cost = np.abs(np.log1p(pred) - np.log1p(truth))
    ax.plot(pred, cost, lw=2, color=color, label=f"true value {truth:,}")

ax.axhline(0.095, color="gray", lw=1, ls="--")
ax.text(300, 0.2, "10% miss = 0.095", color="gray", fontsize=9)
ax.set(xscale="log", ylim=(0, 6), xlabel="prediction (log scale)", ylabel="RMSLE of one prediction",
       title="RMSLE charges for ratios, so small predictions on a true 0 are expensive")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
if FIGURE:
    fig.savefig(FIGURE, dpi=150, bbox_inches="tight")